# 🏠 House Price Prediction - Complete Upgrade

## Features:
- Advanced Feature Engineering
- Multiple ML Models (Linear, Ridge, Lasso, Random Forest, XGBoost, Gradient Boosting)
- Model Comparison with Visualizations
- Interactive Dashboard with Streamlit

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
print('All libraries loaded successfully!')

## 1. Load Data

In [ ]:
df = pd.read_csv('housing.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Data Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Distribution of target
axes[0,0].hist(df['median_house_value'], bins=50, edgecolor='black')
axes[0,0].set_title('House Value Distribution')

# Correlation with income
axes[0,1].scatter(df['median_income'], df['median_house_value'], alpha=0.3)
axes[0,1].set_title('Income vs House Value')

# Rooms vs Value
axes[0,2].scatter(df['total_rooms'], df['median_house_value'], alpha=0.3)
axes[0,2].set_title('Rooms vs House Value')

# Ocean proximity counts
df['ocean_proximity'].value_counts().plot(kind='bar', ax=axes[1,0])
axes[1,0].set_title('Ocean Proximity Distribution')

# Age distribution
axes[1,1].hist(df['housing_median_age'], bins=30, edgecolor='black')
axes[1,1].set_title('Housing Age Distribution')

# Population vs Value
axes[1,2].scatter(df['population'], df['median_house_value'], alpha=0.3)
axes[1,2].set_title('Population vs House Value')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 3. Feature Engineering

In [ ]:
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']
df['income_category'] = pd.cut(df['median_income'], bins=[0, 2, 4, 6, 8, np.inf], 
                               labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])

print('New features created!')
df.head()

## 4. Prepare Data

In [ ]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

numeric_features = ['longitude', 'latitude', 'housing_median_age', 'total_rooms',
                    'total_bedrooms', 'population', 'households', 'median_income',
                    'rooms_per_household', 'bedrooms_per_room', 'population_per_household']
categorical_features = ['ocean_proximity']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Training set: {X_train.shape}')
print(f'Test set: {X_test.shape}')

## 5. Model Training & Comparison

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Lasso Regression': Lasso(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

results = {}

for name, model in models.items():
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')
    
    results[name] = {
        'RMSE': rmse,
        'MAE': mae,
        'R2': r2,
        'CV_R2_Mean': cv_scores.mean(),
        'CV_R2_Std': cv_scores.std()
    }
    
    print(f'{name}:')
    print(f'  RMSE: ${rmse:,.0f}')
    print(f'  MAE: ${mae:,.0f}')
    print(f'  R2 Score: {r2:.4f}')
    print(f'  CV R2 (mean±std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print()

## 6. Model Comparison Visualization

In [ ]:
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# RMSE comparison
results_df['RMSE'].plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('RMSE (Lower is Better)')
axes[0].set_ylabel('RMSE ($)')
axes[0].tick_params(axis='x', rotation=45)

# R2 comparison
results_df['R2'].plot(kind='bar', ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_title('R2 Score (Higher is Better)')
axes[1].set_ylabel('R2')
axes[1].tick_params(axis='x', rotation=45)

# Cross-validation comparison
results_df[['CV_R2_Mean', 'CV_R2_Std']].plot(kind='bar', ax=axes[2], 
                                              yerr='CV_R2_Std', capsize=5,
                                              color='salmon', edgecolor='black')
axes[2].set_title('Cross-Validation R2')
axes[2].set_ylabel('R2')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Results table
results_df.style.format({'RMSE': '${:,.0f}', 'MAE': '${:,.0f}', 
                          'R2': '{:.4f}', 'CV_R2_Mean': '{:.4f}', 'CV_R2_Std': '{:.4f}'})

## 7. Best Model - Feature Importance

In [ ]:
best_model_name = results_df['R2'].idxmax()
print(f'Best Model: {best_model_name}\n')

# Get feature names after preprocessing
ohe_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = list(numeric_features) + list(ohe_features)

best_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', models[best_model_name])
])
best_pipeline.fit(X_train, y_train)

if hasattr(best_pipeline.named_steps['regressor'], 'feature_importances_'):
    importances = best_pipeline.named_steps['regressor'].feature_importances_
    feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=True)
    
    plt.figure(figsize=(10, 6))
    feat_imp.plot(kind='barh')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()

## 8. Save Best Model

In [ ]:
import joblib

joblib.dump(best_pipeline, 'best_house_price_model.pkl')
joblib.dump(list(X.columns), 'model_features.pkl')
print('Model saved successfully!')